# Getting Started with Lexicase Selection

This notebook introduces the lexicase selection library and demonstrates its core functionality.

## Installation

```bash
pip install lexicase
```

In [ ]:
import numpy as np
from lexicase import (
    lexicase_selection,
    epsilon_lexicase_selection,
    downsample_lexicase_selection,
    informed_downsample_lexicase_selection,
)

## Basic Usage

Lexicase selection works with a **fitness matrix** of shape `(n_individuals, n_cases)` where:
- Each row is an individual
- Each column is a test case
- **Higher values = better performance**

In [ ]:
# Create a fitness matrix: 4 individuals evaluated on 3 test cases
fitness_matrix = np.array([
    [10, 5, 8],   # Individual 0
    [8, 9, 6],    # Individual 1
    [6, 7, 9],    # Individual 2
    [4, 3, 7]     # Individual 3
])

print("Fitness matrix shape:", fitness_matrix.shape)
print(fitness_matrix)

In [ ]:
# Select 5 individuals using standard lexicase
selected = lexicase_selection(fitness_matrix, num_selected=5, seed=42)
print("Selected indices:", selected)

Note: Selection is **with replacement** - the same individual can be selected multiple times.

## How Lexicase Works

For each selection:
1. Shuffle the test cases into a random order
2. Start with all individuals as candidates
3. For each test case: keep only candidates with the **best** performance
4. If one candidate remains, select it; otherwise select randomly from survivors

In [ ]:
# Example: specialists on different cases
specialists = np.array([
    [10, 0, 0],   # Best on case 0
    [0, 10, 0],   # Best on case 1
    [0, 0, 10],   # Best on case 2
])

# Each specialist gets selected roughly 1/3 of the time
selected = lexicase_selection(specialists, num_selected=3000, seed=42)
counts = np.bincount(selected, minlength=3)
print("Selection counts:", counts)
print("Proportions:", counts / 3000)

## Epsilon Lexicase Selection

Epsilon lexicase relaxes the strict "best only" filtering. Individuals within `epsilon` of the best are kept as candidates.

By default, epsilon is computed automatically using **Median Absolute Deviation (MAD)** for each test case.

In [ ]:
# Automatic MAD-based epsilon (recommended)
selected = epsilon_lexicase_selection(fitness_matrix, num_selected=5, seed=42)
print("With MAD epsilon:", selected)

In [ ]:
# Manual epsilon value
selected = epsilon_lexicase_selection(fitness_matrix, num_selected=5, epsilon=1.0, seed=42)
print("With epsilon=1.0:", selected)

## Downsampled Lexicase Selection

Uses a random subset of test cases for each selection. Faster and can increase diversity.

In [ ]:
# Large fitness matrix
large_fitness = np.random.rand(100, 50)  # 100 individuals, 50 test cases

# Use only 10 random cases per selection
selected = downsample_lexicase_selection(
    large_fitness, 
    num_selected=20, 
    downsample_size=10, 
    seed=42
)
print("Selected:", selected)

## Elitism

All selection methods support elitism - guaranteeing that the top N individuals (by total fitness) are included.

In [ ]:
fitness = np.array([
    [1, 1, 1],    # Total: 3
    [5, 5, 5],    # Total: 15 (best)
    [2, 2, 2],    # Total: 6
])

# Always include the top 1 individual
selected = lexicase_selection(fitness, num_selected=5, elitism=1, seed=42)
print("Selected with elitism=1:", selected)
print("Individual 1 (best) included:", 1 in selected)

## Reproducibility

Use the `seed` parameter for reproducible results.

In [ ]:
result1 = lexicase_selection(fitness_matrix, num_selected=10, seed=123)
result2 = lexicase_selection(fitness_matrix, num_selected=10, seed=123)

print("Same seed, same result:", np.array_equal(result1, result2))

## Summary

| Function | Use Case |
|----------|----------|
| `lexicase_selection` | Standard lexicase |
| `epsilon_lexicase_selection` | Continuous fitness values |
| `downsample_lexicase_selection` | Many test cases, need speed |
| `informed_downsample_lexicase_selection` | Smart case selection |